# 02 · Operator blocks & identity nesting

omnibias turns the closed-form derivative tower into **typed layers**. Two
ideas:

- **Identity nesting (Lemma 1).** A multi-bias unit (`OMBU`) with tied biases
  and signs that sum to 1 reproduces the base activation **bit-for-bit** — so a
  fresh `OMBU` is a drop-in replacement for any activation, at any `K`.
- **Typed operators.** An `OperatorBlock(op=...)` evaluates a chosen scalar
  operator (`identity` / `grad` / `laplacian` / `integral` / `derivative`) of
  the base activation, using the closed form.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY, WARM, INK
set_style()

from omnibias.torch import OMBU, OperatorBlock, get_activation

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

## 1. Identity nesting is bit-exact

A freshly-initialised `OMBU` uses tied biases and identity-summing signs, so it
returns exactly `σ(z + init_bias)` regardless of `K`. We check this across a
few `K` values — the error is at the float round-off floor.

In [ ]:
z = torch.linspace(-3, 3, 64).unsqueeze(-1)  # (N, 1) -> 1 channel
for base in ["tanh", "sigmoid", "gaussian"]:
    ref = get_activation(base).forward(z)
    for K in [1, 2, 3, 5]:
        ombu = OMBU(num_channels=1, K=K, base=base)
        out = ombu(z)
        err = (out - ref).abs().max().item()
        flag = "identity-nested" if ombu.is_identity_nested else "free"
        print(f"{base:8s} K={K}  max|OMBU - base| = {err:.1e}   ({flag})")

## 2. Typed operator blocks

`OperatorBlock(op="grad")` and `op="laplacian"` evaluate `σ′` and `σ″` of the
base in closed form. We plot the block outputs on top of the analytic
derivatives — they coincide.

In [ ]:
base = "tanh"
spec = get_activation(base)
z = torch.linspace(-4, 4, 200).unsqueeze(-1)

ident = OperatorBlock(op="identity", base=base, channels=1)
grad = OperatorBlock(op="grad", base=base, channels=1)
lap = OperatorBlock(op="laplacian", base=base, channels=1)

y0, y1, y2 = ident(z), grad(z), lap(z)
ref0, ref1, ref2 = spec.fastpath(z, 0), spec.fastpath(z, 1), spec.fastpath(z, 2)
for name, got, ref in [("identity", y0, ref0), ("grad", y1, ref1), ("laplacian", y2, ref2)]:
    print(f"{name:10s} max|block - closed form| = {(got - ref).abs().max().item():.1e}")

zf = z.squeeze(-1).numpy()
fig, ax = plt.subplots()
for lbl, y, col in [("identity σ", y0, INK), ("grad σ′", y1, PRIMARY), ("laplacian σ″", y2, ACCENT)]:
    ax.plot(zf, y.detach().squeeze(-1).numpy(), color=col, label=lbl)
ax.axhline(0, color="#999", lw=0.8)
ax.set_xlabel("z"); ax.set_title(f"OperatorBlock outputs ({base})"); ax.legend(); plt.show()

## Takeaway

`OMBU` is a bit-exact drop-in for an activation; `OperatorBlock` upgrades it
into a **typed differential operator**. The same machinery powers `cmbLinear`
/ `cmbConv*` / `cmbDense` (drop-ins for `Linear` / `Conv` / `Dense`).

Next: **[03 · closed-form-Laplacian PINN](03_pinn_closed_form_laplacian.ipynb)**.